In [2]:
from __future__ import annotations

import argparse
import hashlib
import logging
import os
import random
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Optional
from urllib.parse import urlparse
from typing import Optional
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pymongo import MongoClient, ASCENDING
import requests
from urllib.parse import urljoin, urlparse

In [57]:
LOG_PATH = Path("LOGS/downloader.log")
LOG_PATH.parent.mkdir(exist_ok=True)

logging.basicConfig(
    filename=LOG_PATH,
    filemode="a",
    format=">>> %(levelname)s | %(asctime)s | %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

load_dotenv(override=True)

False

In [58]:
ROOT = "https://www.ola.org/en/legislative-business/house-documents"

client = MongoClient("mongodb://localhost:27017/", tz_aware=True, tzinfo=timezone.utc)
db = client["case-scraping"]
html_col = db["hansard-html-snapshots"]      
pdf_col = db["hansard-pdf-metadata"]         



In [59]:
def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def sleep(min_s: float = 1.0, jitter: float = 2.0) -> None:
    time.sleep(min_s + random.random() * jitter)

In [60]:
# Replace invalid characters with underscores
def safe_filename(s: str) -> str:
    out = []
    for c in s:
        if c.isalnum() or c in (' ', '.', '_', '-'):
            out.append(c)
        else:
            out.append('_')
    return ''.join(out)


# Generate a filename from a URL
def filename_from_url(url: str) -> str:
    p = urlparse(url)
    parts = [x for x in p.path.split("/") if x]
    base = p.netloc + "__" + "__".join(parts) if parts else (p.netloc + "__download")
    #hash of full URL (includes query) for uniqueness of the file name
    h = hashlib.sha256(url.encode("utf-8")).hexdigest()[:8]
    return safe_filename(base) + "__" + h 



In [3]:
# Build proxies from environment variables, returns none if any are missing. A proxy is just a middleman
def build_proxies() -> Optional[dict]:
    proxy = os.getenv("PROXY")
    user = os.getenv("PROXY_USERNAME")
    pw = os.getenv("PROXY_PASSWORD")
    if not (proxy and user and pw):
        return None
    return {scheme: f"http://{user}:{pw}@{proxy}" for scheme in ("http", "https")}


In [4]:
@dataclass
class FetchResult:
    ok: bool
    url: str
    final_url: str
    status: Optional[int]
    content: Optional[bytes]
    headers: Optional[dict]
    error: Optional[str]
    fetched_at: datetime


def fetch(
    url: str,
    proxies: Optional[dict],
    retries: int = 4,
    timeout_s: int = 30,
) -> FetchResult:

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, timeout=timeout_s, proxies=proxies, allow_redirects=True)

            # Retry on certain status codes
            if resp.status_code in (429, 503, 502, 504):
                logger.info("Status %s attempt %s → %s", resp.status_code, attempt, url)
                sleep(10 * attempt, 10)
                continue

            if resp.ok:
                return FetchResult(
                    ok=True,
                    url=url,
                    final_url=str(resp.url),
                    status=resp.status_code,
                    content=resp.content,
                    headers=dict(resp.headers),
                    error=None,
                    fetched_at=utc_now(),
                )

            logger.info("Status %s attempt %s → %s", resp.status_code, attempt, url)
            sleep(5 * attempt, 5)

        except Exception as exc:
            logger.info("Request error attempt %s → %s | %s", attempt, url, exc)
            sleep(5 * attempt, 5)

    return FetchResult(
        ok=False,
        url=url,
        final_url=url,
        status=None,
        content=None,
        headers=None,
        error=f"Failed after {retries} retries",
        fetched_at=utc_now(),
    )

In [63]:
# creates unique indexes on two collections and stops duplicate entries
def ensure_indexes() -> None:
    html_col.create_index([("batch", ASCENDING), ("url", ASCENDING)], unique=True)
    pdf_col.create_index([("batch", ASCENDING), ("pdf_url", ASCENDING)], unique=True)

In [64]:
import re
#this matches the hansard session pages and the hansard day pages
SESSION_RE = re.compile(r"^/en/legislative-business/house-documents/parliament-\d+/session-\d+/?$")
HANSARD_DAY_RE = re.compile(
    r"^/en/legislative-business/house-documents/parliament-\d+/session-\d+/\d{4}-\d{2}-\d{2}/hansard(-\d+)?/?$"
)

In [65]:
def discover_session_urls(root_html: bytes) -> list[str]:
    soup = BeautifulSoup(root_html, "html.parser")
    out = set() #create a set for results to avoid duplicates
    for a in soup.find_all("a", href=True):
        href = a["href"].strip() #gets the clickable link that matches the regex
        if SESSION_RE.match(href):
            out.add(urljoin(ROOT, href))
    return sorted(out)


def discover_hansard_day_urls(session_html: bytes) -> list[str]:
    soup = BeautifulSoup(session_html, "html.parser")
    out = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if HANSARD_DAY_RE.match(href):
            out.add(urljoin(ROOT, href))
    return sorted(out)

In [ ]:
def extract_pdf_links(day_html: bytes, day_url: str) -> list[str]:

    soup = BeautifulSoup(day_html, "html.parser")
    pdfs = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        text = (a.get_text() or "").strip().lower()

        abs_url = urljoin(day_url, href)

        if abs_url.lower().endswith(".pdf"):
            pdfs.add(abs_url)
        elif "pdf" in text and "/pdf" in abs_url.lower():
            pdfs.add(abs_url)

    return sorted(pdfs)


def save_day_html_snapshot(batch: str, day_url: str, r: FetchResult, pdf_links: Optional[list[str]] = None) -> None:
    key = {"batch": batch, "url": day_url}
    doc = {
        **key,
        "final_url": r.final_url,
        "status": r.status,
        "headers": r.headers,
        "fetched_at": r.fetched_at,
        "content": r.content,  # bytes
        "parsed": False,
        "parsed_at": None,
        "parse_error": None,
    }

    if pdf_links is not None:
        doc["pdf_links"] = pdf_links
        doc["pdf_links_updated_at"] = utc_now()

    html_col.update_one(key, {"$set": doc}, upsert=True)



def save_pdf_metadata(batch: str, day_url: str, pdf_url: str, r: FetchResult) -> None:
    key = {"batch": batch, "pdf_url": pdf_url}
    doc = {
        **key,
        "day_url": day_url,
        "final_url": r.final_url,
        "status": r.status,
        "headers": r.headers,
        "fetched_at": r.fetched_at,
        
    }
    pdf_col.update_one(key, {"$set": doc}, upsert=True)


In [67]:
def run(batch: str, limit_sessions: Optional[int] = None, limit_days: Optional[int] = None, download_pdfs: bool = True) -> None:
    proxies = build_proxies()

    logger.info("Fetching root: %s", ROOT)
    root_r = fetch(ROOT, proxies=proxies)
    if not root_r.ok or not root_r.content:
        raise RuntimeError(f"Failed to fetch root: {ROOT}")

    sessions = discover_session_urls(root_r.content)
    logger.info("Discovered %d session URLs", len(sessions))
    if limit_sessions:
        sessions = sessions[:limit_sessions]

    day_urls_all = []
    for i, session_url in enumerate(sessions, start=1):
        sleep(1, 2)
        logger.info("(%d/%d) Fetch session: %s", i, len(sessions), session_url)
        sr = fetch(session_url, proxies=proxies)
        if not sr.ok or not sr.content:
            logger.warning("Failed session: %s", session_url)
            continue
        day_urls = discover_hansard_day_urls(sr.content)
        logger.info("Found %d hansard day links in session", len(day_urls))
        day_urls_all.extend(day_urls)

    # dedupe day URLs
    day_urls_all = sorted(set(day_urls_all))
    logger.info("Total unique hansard day URLs: %d", len(day_urls_all))
    if limit_days:
        day_urls_all = day_urls_all[:limit_days]

    # download each day html + PDFs
    for j, day_url in enumerate(day_urls_all, start=1):
        sleep(1, 2)
        logger.info("(%d/%d) Fetch day: %s", j, len(day_urls_all), day_url)
        dr = fetch(day_url, proxies=proxies)
        if not dr.ok or not dr.content:
            logger.warning("Failed day: %s", day_url)
            continue

        pdf_links = extract_pdf_links(dr.content, day_url)
        logger.info("Found %d PDF links", len(pdf_links))

        save_day_html_snapshot(batch, day_url, dr, pdf_links=pdf_links)


        for pdf_url in pdf_links:
            sleep(1, 2)
            pr = fetch(pdf_url, proxies=proxies, retries=4, timeout_s=60)
            if not pr.ok or not pr.content:
                logger.warning("Failed PDF: %s", pdf_url)
                continue
            save_pdf_metadata(batch, day_url, pdf_url, pr)

    logger.info("DONE batch=%s", batch)

In [68]:
def main(argv=None) -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("batch", help="Batch label (stored on each doc).")
    parser.add_argument("--limit-sessions", type=int, default=None)
    parser.add_argument("--limit-days", type=int, default=None)
    args = parser.parse_args(argv)

    ensure_indexes()
    run(args.batch, args.limit_sessions, args.limit_days)

if __name__ == "__main__":
    try:
        main()
    except Exception:
        logger.exception("Unhandled error")
        raise


usage: ipykernel_launcher.py [-h] [--limit-sessions LIMIT_SESSIONS]
                             [--limit-days LIMIT_DAYS]
                             batch
ipykernel_launcher.py: error: the following arguments are required: batch


SystemExit: 2

In [69]:
main(["my_batch"])
